# Approach 4: VFS_Analyst_v6 Evaluation on 200_gold_standard.csv

Test the same LLM few-shot prompt (VFS_Analyst_v6) that achieved 86.1% on our original test set against a **new 200-article gold standard dataset**.

- Dataset: `200_gold_standard.csv` (200 Alibaba articles, human-annotated 1-5 sentiment)
- Score distribution: 2(1), 3(113), 4(83), 5(3)
- Model: Google Gemini 2.5 Flash Lite
- Prompt: VFS_Analyst_v6 (identical to `LLM_Sentiment_Rating_Fixed.ipynb`)

In [ ]:
# Step 1: Imports & Configuration
import os, json, re, time
import pandas as pd
import numpy as np
from google import genai
from google.genai import types
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

DEFAULT_MODEL = 'gemini-2.5-flash-lite'
MAX_RETRIES   = 5
RATE_DELAY    = 0.4
DATA_FILE     = 'data/200_gold_standard.csv'
OUTPUT_DIR    = 'output'
CACHE_CSV     = os.path.join(OUTPUT_DIR, '200_gold_VFS_Analyst_v6.csv')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Config loaded')

Config loaded


In [ ]:
# Step 2: Gemini Client
class GeminiLLMClient:
    def __init__(self, api_key, model=DEFAULT_MODEL):
        self.client = genai.Client(api_key=api_key)
        self.model = model
        self.request_count = 0

    def classify_sentiment(self, title, body, system_prompt, user_prompt):
        message = user_prompt.format(title=title, body=body)
        for attempt in range(MAX_RETRIES):
            try:
                resp = self.client.models.generate_content(
                    model=self.model,
                    contents=message,
                    config=types.GenerateContentConfig(
                        system_instruction=system_prompt,
                        temperature=0.3,
                        max_output_tokens=500,
                    )
                )
                self.request_count += 1
                return {'success': True, 'content': resp.text}
            except Exception as e:
                err = str(e)[:120]
                wait = 20 + attempt * 10 if 'location' in err.lower() else 2 ** attempt
                if attempt < MAX_RETRIES - 1:
                    print(f'   Retry {attempt+1}/{MAX_RETRIES}: {err[:60]}... waiting {wait}s')
                    time.sleep(wait)
                else:
                    return {'success': False, 'error': err}
        return {'success': False, 'error': 'max retries'}

API_KEY = os.getenv('GEMINI_API_KEY')
if not API_KEY:
    raise ValueError('Set GEMINI_API_KEY before running this notebook')
llm_client = GeminiLLMClient(API_KEY)
print('Gemini client ready')

Gemini client ready


In [ ]:
# Step 3: Build VFS_Analyst_v6 Prompt
# Few-shot examples from ORIGINAL training set (alibaba_sentiment_merged.xlsx)
from sklearn.model_selection import train_test_split

df_orig = pd.read_excel('data/alibaba_sentiment_merged.xlsx', engine='openpyxl')
df_orig = df_orig.rename(columns={'original_title_en': 'title_en', 'original_body_en': 'body_en'})
df_orig['sentiment_reason'] = df_orig['sentiment_reason'].fillna('')
df_orig = df_orig[df_orig['sentiment_score'].notna()].copy()
df_orig['sentiment_score'] = df_orig['sentiment_score'].astype(int)
df_orig = df_orig[df_orig['sentiment_score'].between(1, 5)].reset_index(drop=True)

_min_cls = df_orig['sentiment_score'].value_counts().min()
_strat = df_orig['sentiment_score'] if _min_cls >= 3 else None
try:
    _tr, _tmp = train_test_split(df_orig, test_size=0.4, random_state=42, stratify=_strat)
except ValueError:
    _tr, _tmp = train_test_split(df_orig, test_size=0.4, random_state=42)
fs_train = _tr.reset_index(drop=True)

def build_fs_block(df_train, n_per_class=2, body_len=350, seed=42):
    lines = []
    for score in sorted(df_train['sentiment_score'].unique()):
        subset = df_train[df_train['sentiment_score'] == score]
        sample = subset.sample(min(n_per_class, len(subset)), random_state=seed)
        for _, row in sample.iterrows():
            lines.append(
                f'---\n'
                f'Title: {str(row.get("title_en",""))[:120].strip()}\n'
                f'Body (excerpt): {str(row.get("body_en",""))[:body_len].strip()}\n'
                f'Human score: {int(row["sentiment_score"])}\n'
                f'Human reasoning: {str(row.get("sentiment_reason",""))[:200].strip()}'
            )
    return '\n'.join(lines)

FS_BLOCK = build_fs_block(fs_train)

USER_TMPL = (
    'Analyze this Alibaba news article and classify its sentiment:\n\n'
    '**Title**: {title}\n\n**Body**: {body}\n\n'
    'Return ONLY the JSON object as specified, no other text.'
)

SYS_PROMPT = (
    'You are a sell-side financial analyst covering Alibaba Group.\n'
    'Rate news articles on a 1-5 sentiment scale from an investor\'s perspective:\n\n'
    '1 = strongly negative  (major crisis directly involving Alibaba)\n'
    '2 = negative           (specific negative impact on Alibaba\'s business)\n'
    '3 = neutral            (general context, mixed signals, or indirect connection)\n'
    '4 = positive           (specific positive impact on Alibaba\'s business or outlook)\n'
    '5 = strongly positive  (exceptional achievement or market-leadership win for Alibaba)\n\n'
    '--- SCORE 2 CALIBRATION ---\n'
    'Rate 2 (negative) when any of the following apply:\n'
    '  a) Alibaba is NAMED as facing a penalty, investigation, revenue loss, or market setback\n'
    '  b) NEW RULES specifically targeting e-commerce platforms, online marketplaces,\n'
    '     cross-border e-commerce, or internet platform businesses -- including\n'
    '     NEW TAX/COMPLIANCE OBLIGATIONS on platform operators or merchants\n'
    '  c) Competitor gains explicitly described as coming at Alibaba\'s expense\n'
    'Rate 3 (NOT 2) for:\n'
    '  - Broad antitrust enforcement affecting many industries (general fair-competition news)\n'
    '  - Supply-chain security decrees, physical logistics rules, general trade regulations\n'
    '  - Consumer protection or sector rules that do NOT target the platform/e-commerce model\n'
    '  - Vague macro headwinds without direct platform-economy impact\n\n'
    '--- SCORE 4 CALIBRATION ---\n'
    'Rate 4 (positive) when any of the following apply:\n'
    '  a) Direct Alibaba development: product launch, revenue growth, partnership, strategic win\n'
    '  b) Policy explicitly supporting e-commerce consumption, cloud computing, AI infrastructure,\n'
    '     or digital economy in a way that directly benefits Alibaba\'s revenue streams\n'
    '  c) AI INFRASTRUCTURE: new GPU/chip generation announcements (e.g. NVIDIA GTC, new\n'
    '     accelerator chips) -- these directly expand Alibaba Cloud\'s GPU/AI service capacity\n'
    '  d) CHINESE AI ECOSYSTEM news: domestic AI usage milestones (token call volumes,\n'
    '     Chinese LLM rankings), Chinese tech companies (including Alibaba) investing heavily\n'
    '     in AI infrastructure -- Alibaba Tongyi/Qianwen is a TOP beneficiary\n'
    '  e) Alibaba\'s senior leadership (Jack Ma, CEO) making a strategic move reported as\n'
    '     significant standalone or headline news\n'
    '  f) Multi-company tech roundup (Tech Weekly, AI Weekly, market briefing) where one of\n'
    '     the KEY ITEMS is a MEANINGFUL Alibaba development (chip, AI model, investment,\n'
    '     stock movement, subsidiary win)\n'
    'Rate 3 (NOT 4) for:\n'
    '  - Purely foreign AI company news (OpenAI, Google, Meta) with no Chinese AI angle\n'
    '  - Broad economic growth or service-industry policies benefiting many sectors equally\n'
    '  - Roundup articles where Alibaba receives only a BRIEF or PERIPHERAL mention\n'
    '    alongside 5+ other companies with no significant Alibaba-specific item\n\n'
    '--- SCORE 5 CALIBRATION ---\n'
    'Rate 5 (strongly positive) when Alibaba achieves clear MARKET LEADERSHIP:\n'
    '  - Alibaba\'s AI product dominates a major high-profile competitive event\n'
    '    (e.g. winning AI red envelope war by volume, topping a national AI benchmark)\n'
    '  - Record-breaking Alibaba financial results or landmark regulatory/competitive win\n\n'
    '--- FEW-SHOT EXAMPLES ---\n\n'
    f'{FS_BLOCK}\n\n'
    '--- SCORING PROCESS ---\n'
    '  1. What is the article\'s main development? (one sentence)\n'
    '  2. Score-2 check: platform/e-commerce-specific rule, named Alibaba setback?\n'
    '  3. Score-4 check: AI infrastructure, Chinese AI milestone, direct Alibaba positive,\n'
    '     or roundup with meaningful Alibaba item?\n'
    '  4. If neither -> score 3; if one fires -> 2 or 4\n'
    '  5. If positive is EXCEPTIONAL (market leadership win) -> score 5\n\n'
    'Return ONLY valid JSON: {"sentiment_score":<1-5>,"confidence":<0-1>,'
    '"key_factors":["..."],"reasoning":"..."}'
)

print(f'Prompt built (system prompt length: {len(SYS_PROMPT):,} chars)')
print(f'Few-shot examples from: alibaba_sentiment_merged.xlsx (train split, n={len(fs_train)})')

Prompt built (system prompt length: 9,469 chars)
Few-shot examples from: alibaba_sentiment_merged.xlsx (train split, n=105)


In [9]:
# Step 4: Run Evaluation on 200_gold_standard.csv
# Results cached to CSV. Delete cache file to force re-evaluation.

def parse_response(resp):
    if not resp.get('success', False):
        return {'pred': None, 'conf': 0.0, 'ok': False, 'reasoning': resp.get('error', '')[:80]}
    try:
        txt = re.sub(r'```(?:json)?\s*', '', resp['content']).strip().rstrip('`')
        try:
            d = json.loads(txt)
        except json.JSONDecodeError:
            m = re.search(r'\{.*?\}', txt, re.DOTALL)
            d = json.loads(m.group()) if m else {}
        score = max(1, min(5, int(d.get('sentiment_score', 3))))
        conf = max(0.0, min(1.0, float(d.get('confidence', 0.5))))
        return {'pred': score, 'conf': conf, 'ok': True,
                'reasoning': str(d.get('reasoning', ''))[:150]}
    except Exception as e:
        return {'pred': None, 'conf': 0.0, 'ok': False, 'reasoning': str(e)[:80]}

df_200 = pd.read_csv(DATA_FILE)
df_200['article_id'] = df_200.index + 1
print(f'Loaded {len(df_200)} articles from {DATA_FILE}')
print(f'Score distribution: {df_200["sentiment_score"].value_counts().sort_index().to_dict()}')

if os.path.exists(CACHE_CSV):
    df_results = pd.read_csv(CACHE_CSV)
    print(f'\nLoaded cached results: {CACHE_CSV} ({len(df_results)} rows)')
else:
    print(f'\nRunning LLM evaluation on {len(df_200)} articles ...')
    rows = []
    t0 = time.time()
    for i, (_, row) in enumerate(df_200.iterrows()):
        resp = llm_client.classify_sentiment(
            str(row['title_en']), str(row['body_en']),
            SYS_PROMPT, USER_TMPL)
        parsed = parse_response(resp)
        rows.append({
            'article_id': row['article_id'],
            'title': str(row['title_en'])[:70],
            'gold': int(row['sentiment_score']),
            'pred': parsed['pred'],
            'conf': parsed['conf'],
            'ok': parsed['ok'],
            'reason': parsed['reasoning']
        })
        if (i+1) % 20 == 0 or i == len(df_200)-1:
            elapsed = time.time() - t0
            print(f'   {i+1}/{len(df_200)} | {elapsed:.0f}s')
        if i < len(df_200) - 1:
            time.sleep(RATE_DELAY)

    df_results = pd.DataFrame(rows)
    df_results.to_csv(CACHE_CSV, index=False, encoding='utf-8-sig')
    print(f'\nSaved to {CACHE_CSV}')

ok_count = (df_results['ok'].astype(str).str.lower() == 'true').sum()
print(f'\nSuccessfully scored: {ok_count}/{len(df_results)}')

Loaded 200 articles from 200_gold_standard.csv
Score distribution: {2: 1, 3: 113, 4: 83, 5: 3}

Loaded cached results: llm_sentiment_output\200_gold_VFS_Analyst_v6.csv (200 rows)

Successfully scored: 200/200


In [10]:
# Step 5: Metrics & Comparison
df_eval = df_results[df_results['ok'].astype(str).str.lower() == 'true'].copy()
df_eval['gold'] = df_eval['gold'].astype(int)
df_eval['pred'] = df_eval['pred'].astype(int)

n = len(df_eval)
exact = (df_eval['pred'] == df_eval['gold']).mean()
adj   = (abs(df_eval['pred'] - df_eval['gold']) <= 1).mean()
mae   = abs(df_eval['pred'] - df_eval['gold']).mean()
rho, pval = stats.spearmanr(df_eval['pred'], df_eval['gold'])

print('=' * 68)
print('  VFS_Analyst_v6 -- Evaluation on 200_gold_standard.csv')
print('=' * 68)
print(f'  Exact match accuracy : {exact*100:.1f}%')
print(f'  +/-1 accuracy        : {adj*100:.1f}%')
print(f'  MAE                  : {mae:.3f}')
print(f'  Spearman rho         : {rho:.3f}  (p={pval:.4f})')
print(f'  n                    : {n}')
print()
print('  -- Comparison with original dataset --')
print(f'{"Dataset":<30} {"Exact":>8} {"+/-1":>8} {"MAE":>7} {"rho":>7} {"n":>5}')
print(f'  {"-"*65}')
print(f'  {"Original test set":<30} {"86.1%":>8} {"100%":>8} {"0.167":>7} {"0.904":>7} {"36":>5}')
print(f'  {"Original val set":<30} {"82.9%":>8} {"100%":>8} {"0.200":>7} {"0.905":>7} {"35":>5}')
print(f'  {"200_gold_standard":<30} {exact*100:>7.1f}% {adj*100:>7.1f}% {mae:>7.3f} {rho:>7.3f} {n:>5}')
print('=' * 68)

# Confusion matrix
print('\n  Confusion Matrix (gold rows x pred cols):')
ct = pd.crosstab(df_eval['gold'], df_eval['pred'], rownames=['gold'], colnames=['pred'])
print(ct.to_string())

# Per-class accuracy
print('\n  Per-class accuracy:')
for score in sorted(df_eval['gold'].unique()):
    cls = df_eval[df_eval['gold'] == score]
    acc = (cls['pred'] == cls['gold']).mean() * 100
    print(f'    Score {score}: {acc:.0f}%  ({(cls["pred"]==cls["gold"]).sum()}/{len(cls)})  '
          f'pred dist = {cls["pred"].value_counts().sort_index().to_dict()}')

# Score distribution comparison
print('\n  Score distribution:')
print(f'    Gold: {df_eval["gold"].value_counts().sort_index().to_dict()}')
print(f'    Pred: {df_eval["pred"].value_counts().sort_index().to_dict()}')

  VFS_Analyst_v6 -- Evaluation on 200_gold_standard.csv
  Exact match accuracy : 69.0%
  +/-1 accuracy        : 100.0%
  MAE                  : 0.310
  Spearman rho         : 0.540  (p=0.0000)
  n                    : 200

  -- Comparison with original dataset --
Dataset                           Exact     +/-1     MAE     rho     n
  -----------------------------------------------------------------
  Original test set                 86.1%     100%   0.167   0.904    36
  Original val set                  82.9%     100%   0.200   0.905    35
  200_gold_standard                 69.0%   100.0%   0.310   0.540   200

  Confusion Matrix (gold rows x pred cols):
pred  2    3   4
gold            
2     1    0   0
3     7  104   2
4     0   50  33
5     0    0   3

  Per-class accuracy:
    Score 2: 100%  (1/1)  pred dist = {2: 1}
    Score 3: 92%  (104/113)  pred dist = {2: 7, 3: 104, 4: 2}
    Score 4: 40%  (33/83)  pred dist = {3: 50, 4: 33}
    Score 5: 0%  (0/3)  pred dist = {4: 3}

  S